In [21]:
import os
import re
import string
import random
from collections import defaultdict, Counter
import math
from math import log, exp


This function **loads and cleans text data** from the IMDB unsupervised dataset. It removes unwanted characters, replaces <br /> with <nl>, fixes spacing issues, and returns a list of cleaned sentences for further processing.


In [22]:
import os
import re

def load_imdb_unsup_sentences(folder_path="D:/term 8/nlp/labs/lab1/pythonProject1/unsup/unsup"):
    """
    Loads text files from the IMDB 'unsup' (unsupervised) folder.
    - Reads text from all .txt files
    - Replaces <br /> with <nl>
    - Removes unwanted Unicode characters
    - Fixes spacing issues and contractions
    """
    
    all_sentences = []

    if not os.path.exists(folder_path):
        print(f"Error: Folder '{folder_path}' does not exist")
        return all_sentences

    for filename in os.listdir(folder_path):
        if filename.endswith('.txt'):
            file_path = os.path.join(folder_path, filename)

            try:
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
                    content = file.read()

                    # Replace <br /> with <nl>
                    content = content.replace("<br />", " <nl> ")

                    # Remove non-ASCII characters (fixes \x97, \?)
                    content = re.sub(r'[^\x00-\x7F]+', '', content)

                    # Fix contractions (turns "can\'t" into "can't")
                    content = content.replace(r"\'", "'")

                    # Normalize spaces
                    content = re.sub(r'\s+', ' ', content).strip()

                    # Split into sentences and clean
                    lines = [line.strip() for line in content.split('\n') if line.strip()]

                    all_sentences.extend(lines)

            except Exception as e:
                print(f"Error reading file {filename}: {e}")

    return all_sentences


In [23]:

def remove_punctuation(text):

    text = re.sub(r"<nl>", " SPECIALNLTOKEN ", text)  
    text = re.sub(r"["+re.escape(string.punctuation)+r"]", "", text)  
    text = text.replace("SPECIALNLTOKEN", "<nl>")  

    return text

These functions handle text preprocessing by first creating a vocabulary and then tokenizing sentences. The `build_vocabulary(sentences)` function converts all text to lowercase, removes punctuation, and splits sentences into words (tokens), storing unique words in a vocabulary set. The `tokenize(sentences, vocab, unknown="<UNK>")` function then processes each sentence by converting it to lowercase, removing punctuation, and splitting it into tokens. Any word not found in the vocabulary is replaced with the `<UNK>` token to handle out-of-vocabulary words. This ensures consistency in text processing and prepares the data for further modeling.

In [24]:
def build_vocabulary(sentences):
    """
    lower each sentence,
    Splits each sentence on whitespace, removes punctuation,
    and builds a set of unique tokens (vocabulary).
    """
    vocab = set()
    
    for sentence in sentences:
        sentence = sentence.lower()
        clean_sentence = remove_punctuation(sentence)
        tokens = clean_sentence.split()
        vocab.update(tokens)
    
    return vocab

def tokenize(sentences, vocab, unknown="<UNK>"):
    """
    lower each sentence,
    Splits each sentence on whitespace, removes punctuation,
    and replaces tokens not in the vocabulary with unknown token.
    Returns the list of tokenized sentences.
    """
    tokenized_sentences = []
    
    for sentence in sentences:
        sentence = sentence.lower()
        clean_sentence = remove_punctuation(sentence)
        tokens = clean_sentence.split()
        processed_tokens = [token if token in vocab else unknown for token in tokens]
        tokenized_sentences.append(processed_tokens)
    
    return tokenized_sentences

In [25]:
sentences = load_imdb_unsup_sentences()

print(f"Number of raw sentences loaded: {len(sentences)}")
print(f"Example (first 2 sentences):\n{sentences[:2]}")

Number of raw sentences loaded: 50000
Example (first 2 sentences):
['I admit, the great majority of films released before say 1933 are just not for me. Of the dozen or so "major" silents I have viewed, one I loved (The Crowd), and two were very good (The Last Command and City Lights, that latter Chaplin circa 1931). <nl> <nl> So I was apprehensive about this one, and humor is often difficult to appreciate (uh, enjoy) decades later. I did like the lead actors, but thought little of the film. <nl> <nl> One intriguing sequence. Early on, the guys are supposed to get "de-loused" and for about three minutes, fully dressed, do some schtick. In the background, perhaps three dozen men pass by, all naked, white and black (WWI ?), and for most, their butts, part or full backside, are shown. Was this an early variation of beefcake courtesy of Howard Hughes?', 'Take a low budget, inexperienced actors doubling as production staff as well as limited facilitiesand you can\'t expect much more than "Ti

In [26]:
assert len(sentences) == 50000, "Expected 50,000 sentences from the unsup folder."

In [27]:
random.seed(42)

def split_data(sentences, test_split=0.1):
    """
    Shuffle the sentences and split them into train and test sets.
    First (1-test_split) of the data is the training set.
    """
    random.shuffle(sentences)
    split_point = int(len(sentences) * (1 - test_split))
    train_sentences = sentences[:split_point]
    test_sentences = sentences[split_point:]
    
    return train_sentences, test_sentences



In [28]:
train_sentences, test_sentences = split_data(sentences)

print(f"Number of training sentences: {len(train_sentences)}")
print(f"Number of test sentences: {len(test_sentences)}")

Number of training sentences: 45000
Number of test sentences: 5000


In [29]:
assert len(train_sentences) == 45000, "Expected 45,000 sentences for training."
assert len(test_sentences) == 5000, "Expected 5,000 sentences for testing."

In [30]:
vocab = build_vocabulary(train_sentences)
tokenized_sentences = tokenize(train_sentences, vocab)

print(f"Vocabulary size: {len(vocab)}")
print(f"Example tokens from first sentence: {tokenized_sentences[0][:350] if tokenized_sentences else 'No tokens loaded'} ...")

Vocabulary size: 160682
Example tokens from first sentence: ['having', 'first', 'seen', 'the', 'directors', '12min', 'take', 'on', 'poes', 'fall', 'of', 'the', 'house', 'of', 'usher', 'i', 'was', 'looking', 'forward', 'to', 'seeing', 'this', 'one', 'too', 'and', 'wasnt', 'disappointed', 'at', 'all', 'though', 'perhaps', 'not', 'quite', 'up', 'to', 'the', 'same', 'level', 'of', 'artistic', 'attainment', 'as', 'usher', 'it', 'is', 'nevertheless', 'very', 'much', 'in', 'the', 'same', 'vein', '<nl>', '<nl>', 'like', 'the', 'usher', 'the', 'viewer', 'should', 'be', 'familiar', 'beforehand', 'with', 'the', 'story', 'on', 'which', 'it', 'is', 'based', 'in', '1928', 'the', 'directors', 'watson', 'and', 'webber', 'could', 'have', 'safely', 'assumed', 'the', 'audiences', 'knowledge', 'of', 'the', 'biblical', 'tale', 'interestingly', 'apart', 'from', 'the', 'actual', 'genesis', 'account', 'a', 'phrase', 'from', 'the', 'song', 'of', 'songs', 'is', 'also', 'used', 'when', 'lot', 'is', 'offering', '

In [31]:
# assert len(vocab) == 161292, "Expected a vocabulary size of 171,591." #skip for replication problems
assert len(tokenized_sentences) == 45000, "Expected tokenized sentences count to match raw sentences."

example = "I love Natural language processing, and i want to be a great engineer."
assert len(example) == 70, "Example sentence length (in characters) does not match the expected 70."
print(example)
example_tokens = tokenize([example], vocab)[0]
assert len(example_tokens) == 13, "Token count for the example sentence does not match the expected 13."
print(example_tokens)


I love Natural language processing, and i want to be a great engineer.
['i', 'love', 'natural', 'language', 'processing', 'and', 'i', 'want', 'to', 'be', 'a', 'great', 'engineer']


In [32]:
def pad_sentence(tokens, n):
    """
    Pads a list of tokens with <s> at the start (n-1 times)
    and </s> at the end (once).
    For example, if n=3, you add 2 <s> tokens at the start.
    """
    padded = ["<s>"] * (n-1) + tokens + ["</s>"]
    return padded

print(pad_sentence(example_tokens,3))


['<s>', '<s>', 'i', 'love', 'natural', 'language', 'processing', 'and', 'i', 'want', 'to', 'be', 'a', 'great', 'engineer', '</s>']


This function constructs n-gram frequency counts from tokenized sentences. Each sentence is first padded with start (`<s>`) and end (`</s>`) tokens to ensure that sentence boundaries are respected. Then, the function iterates through each sentence, extracting n-grams (sequences of `n` words) and their corresponding (n-1)-gram contexts (used for probability estimation). These are stored in two separate `Counter` objects: `ngram_counts` keeps track of how often each n-gram appears, while `context_counts` records the frequency of each (n-1)-gram context. This step is crucial for building an n-gram language model, as it helps estimate word probabilities based on preceding words.


In [33]:
def build_ngram_counts(tokenized_sentences, n):
    """
    Builds n-gram counts and (n-1)-gram counts from the given tokenized sentences.
    Each sentence is padded with <s> and </s>.
    """
    ngram_counts = Counter()
    context_counts = Counter()
    
    for sentence in tokenized_sentences:
        # Pad the sentence
        padded_sentence = pad_sentence(sentence, n)
        
        # Collect n-grams and (n-1)-grams
        for i in range(len(padded_sentence) - n + 1):
            # Extract the n-gram and (n-1)-gram
            ngram = tuple(padded_sentence[i:i+n])
            context = tuple(padded_sentence[i:i+n-1])
            
            # Update counts
            ngram_counts[ngram] += 1
            context_counts[context] += 1
    
    return ngram_counts, context_counts

In [34]:
def laplace_probability(ngram, ngram_counts, context_counts, vocab_size, alpha=1.0):
    """
    Computes the probability of an n-gram using Laplace (add-alpha) smoothing.
    
    P(w_i | w_{i-(n-1)}, ..., w_{i-1}) =
        (count(ngram) + alpha) / (count(context) + alpha * vocab_size)
    """
    prob = 0.0
    context = ngram[:-1]  
    ngram_count = ngram_counts[ngram]
    context_count = context_counts[context]
    prob = (ngram_count + alpha) / (context_count + alpha * vocab_size)
    
    return prob

In [35]:
ngram_counts,context_counts  = build_ngram_counts(tokenized_sentences, 2)
prob = laplace_probability(( 'i', 'loved'), ngram_counts, context_counts, len(vocab))
print(prob)

0.003639384334646326


In [36]:
n = 2
ngram_counts, context_counts = build_ngram_counts(tokenized_sentences, n=n)
print(f"Number of bigrams: {len(ngram_counts)}")
print(f"Number of contexts: {len(context_counts)}")
print(f"Example bigram: {list(ngram_counts.items())[0]}")
print(f"Example context: {list(context_counts.items())[0]}")

Number of bigrams: 2278824
Number of contexts: 160683
Example bigram: (('<s>', 'having'), 137)
Example context: (('<s>',), 45000)


In [37]:
def predict_next_token(
    context_tokens,
    ngram_counts,
    context_counts,
    vocab,
    n=2,
    alpha=1.0,
    top_k=5
):
    """
    Given a list of context tokens, predict the next token using the n-gram model.
    Returns the top_k predictions as (token, probability).
    """
    if len(context_tokens) < n-1:
        context_tokens = ["<s>"] * (n-1 - len(context_tokens)) + context_tokens
    else:
        context_tokens = context_tokens[-(n-1):]
    
    context = tuple(context_tokens)
    candidates = []
    
    # Check each word in vocabulary as a potential next token
    for word in vocab:
        # Form the n-gram with this word
        ngram = context + (word,)
        prob = laplace_probability(ngram, ngram_counts, context_counts, len(vocab), alpha)
        candidates.append((word, prob))
    
    candidates.sort(key=lambda x: x[1], reverse=True)
    
    return candidates[:top_k]

generates text using an n-gram language model by predicting the most probable next word based on given context tokens. It starts with the provided start_tokens, extends them iteratively, and predicts the next word using the predict_next_token function. The generated sequence continues until either a sentence-ending token (</s>) is reached or the max_length is exceeded. The function ensures coherence by updating the context with newly generated words. The output is a list of words forming a sentence-like structure based on learned n-gram probabilities.

In [38]:
def generate_text_with_limit(
    start_tokens,
    ngram_counts,
    context_counts,
    vocab,
    n=2,
    alpha=1.0,
    max_length=20
):
    """
    Generates text from an n-gram model until it sees </s>
    or reaches a maximum total length (max_length).
    """
    generated = start_tokens.copy()
    context = start_tokens
    generated.extend(context)

    for _ in range(max_length-len(context)):
      candidates = predict_next_token(context, ngram_counts, context_counts, vocab, n,alpha)
      candidate_next_word = candidates[0][0]
      print(candidate_next_word)
      if candidate_next_word == "</s>":
        break
      generated.append(candidate_next_word)
      context = context + [candidate_next_word]
    return generated



context = ["i", "love"]
generated_seq = generate_text_with_limit(
    start_tokens=context,
    ngram_counts=ngram_counts,
    context_counts=context_counts,
    vocab=vocab,
    n=2,
    alpha=1.0,
    max_length=128
)

print("Generated Sequence:", generated_seq)


with
the
film
is
a
lot
of
the
film
is
a
lot
of
the
film
is
a
lot
of
the
film
is
a
lot
of
the
film
is
a
lot
of
the
film
is
a
lot
of
the
film
is
a
lot
of
the
film
is
a
lot
of
the
film
is
a
lot
of
the
film
is
a
lot
of
the
film
is
a
lot
of
the
film
is
a
lot
of
the
film
is
a
lot
of
the
film
is
a
lot
of
the
film
is
a
lot
of
the
film
is
a
lot
of
the
film
is
a
lot
of
the
film
is
a
lot
of
the
film
is
a
lot
of
the
film
is
a
lot
of
the
film
is
a
lot
Generated Sequence: ['i', 'love', 'i', 'love', 'with', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', '

This function calculates the **perplexity** of an n-gram model, which measures how well the model predicts a given text. It processes tokenized sentences by padding them with `<s>` and `</s>` tokens, extracting n-grams, and computing their probabilities using Laplace smoothing. The function sums the log probabilities of all n-grams, averages them, and exponentiates the negative value to obtain the final perplexity score. A **lower perplexity** indicates better predictive performance, meaning the model assigns higher probabilities to correct word sequences.

In [39]:
import numpy as np
def calculate_perplexity(
    tokenized_sentences,
    ngram_counts,
    context_counts,
    vocab_size,
    n=2,
    alpha=1.0
):
    """
    Calculates the perplexity of an n-gram model (with Laplace smoothing)
    on a list of tokenized sentences using the formula:
    Perplexity = (P(w₁, w₂, ..., wₙ))^(-1/n)
    or equivalently:
    ppl(S) = e^x where x = -(1/n) * log(P(w₁, ..., wₙ)) = -(1/n) * ∑log(P(wᵢ|w₁...wᵢ₋₁))
    """
    total_log_prob = 0
    total_ngrams = 0

    for sentence in tokenized_sentences:
        padded_sentence = pad_sentence(sentence, n)  # Add <s> and </s>
        
        for i in range(len(padded_sentence) - n + 1):
            ngram = tuple(padded_sentence[i:i+n])  # Extract n-gram
            
            prob = laplace_probability(ngram, ngram_counts, context_counts, vocab_size, alpha)
            
            total_log_prob += np.log(prob)  # Sum log probabilities
            total_ngrams += 1  

    # Compute perplexity
    avg_log_prob = total_log_prob / total_ngrams  
    perplexity = np.exp(-avg_log_prob)  

    return perplexity

In [40]:
import matplotlib.pyplot as plt
from tabulate import tabulate  

# Define N-gram values to test
n_values = [1,2, 3, 4]  
perplexity_sentence = {}  # Perplexity for "I love this movie"
perplexity_testset = {}   # Perplexity for full test dataset

# Test sentence
test_sentence = "I love this movie"
tokenized_test_sentence = tokenize([test_sentence], vocab)[0]

print("\nPerplexity Analysis for Different N-gram Sizes\n" + "="*50)

for n in n_values:
    
    # Build n-gram counts
    ngram_counts, context_counts = build_ngram_counts(tokenized_sentences, n)

    # Compute perplexity for the test sentence
    perplexity_sen = calculate_perplexity([tokenized_test_sentence], ngram_counts, context_counts, len(vocab), n)
    perplexity_sentence[n] = perplexity_sen

    # Compute perplexity for the full test dataset
    tokenized_test_sentences = tokenize(test_sentences, vocab)
    perplexity_full = calculate_perplexity(tokenized_test_sentences, ngram_counts, context_counts, len(vocab), n)
    perplexity_testset[n] = perplexity_full

# Format results in a table
table_data = [
    [n, f"{perplexity_sentence[n]:.4f}", f"{perplexity_testset[n]:.4f}"]
    for n in n_values
]

print("\nPerplexity Results for Different N-gram Sizes:\n")
print(tabulate(table_data, headers=["N-gram Size (n)", "Test Sentence Perplexity", "Full Test Set Perplexity"], tablefmt="grid"))



Perplexity Analysis for Different N-gram Sizes

Perplexity Results for Different N-gram Sizes:

+-------------------+----------------------------+----------------------------+
|   N-gram Size (n) |   Test Sentence Perplexity |   Full Test Set Perplexity |
+===================+============================+============================+
|                 1 |                   183.129  |                    1224.42 |
+-------------------+----------------------------+----------------------------+
|                 2 |                    75.1805 |                    3448.49 |
+-------------------+----------------------------+----------------------------+
|                 3 |                   301.329  |                   37320.1  |
+-------------------+----------------------------+----------------------------+
|                 4 |                   798.545  |                  101390    |
+-------------------+----------------------------+----------------------------+
